In [3]:
from embedder import Embedder

embed = Embedder()

In [4]:
q1 = "How does approximate nearest neighbor search work?"

In [5]:
v = embed.encode(q1)

In [6]:
#Answer 1
v[0]

np.float64(-0.020582034609969147)

In [7]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [63]:
for doc in documents:
    if doc['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md':
        dv = embed.encode(doc['content'])

In [62]:
# Answer 2
v.dot(dv)

np.float64(0.3610703009158738)

In [12]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [20]:
import numpy as np
X = []
for i in chunks:
    batch_vectors = embed.encode_batch([i['filename'] + i['content']])
    X.extend(batch_vectors)
X = np.array(X)

In [22]:
scores = X.dot(v)
idx = np.argmax(scores)

In [64]:
#Answer 3
chunks[idx]['filename']

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [26]:
from minsearch import VectorSearch
vindex = VectorSearch()
vindex.fit(X, chunks)

In [34]:
q2 = 'What metric do we use to evaluate a search engine?'

In [35]:
v2 = embed.encode(q2)
results = vindex.search(v2, num_results=5)

In [36]:
#Answer 4
results[0]['filename']

'04-evaluation/lessons/05-search-metrics.md'

In [37]:
from minsearch import Index

In [38]:
index = Index(text_fields=["content"])
index.fit(chunks)

In [39]:
q3 = 'How do I store vectors in PostgreSQL?'

In [41]:
v3 = embed.encode(q3)
results_v3 = vindex.search(v3, num_results=5)

In [43]:
results_t3 = index.search(q3, num_results=5)

In [48]:
#Answer 5
set(fn['filename'] for fn in results_v3) - set(fn['filename'] for fn in results_t3)

{'02-vector-search/lessons/08-pgvector.md'}

In [49]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [50]:
q4 = 'How do I give the model access to tools?'

In [56]:
v4 = embed.encode(q4)
vector_results = vindex.search(v4)

In [57]:
text_results = index.search(q4)

In [58]:
results = rrf([vector_results, text_results])

In [65]:
#Answer 6
results[0]['filename']

'01-agentic-rag/lessons/13-function-calling.md'